In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans

# --- Đường dẫn ảnh ---
foreground_path = "balls.png"               # ảnh gốc cần tách nền
background_path = "background.jpg"  # ảnh nền mới
# output_path = "replace_background_kmeans.jpg"

# --- Đọc ảnh ---
img = cv2.imread(foreground_path)
h, w, _ = img.shape
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# --- Chuẩn bị dữ liệu cho KMeans ---
pixels = img_rgb.reshape(-1, 3)
coords = np.indices((h, w)).transpose(1, 2, 0).reshape(-1, 2)
X = np.hstack([pixels, coords / np.array([[h, w]]) * 255])  # thêm vị trí để tách chính xác hơn

# --- Chạy KMeans ---
k = 2  # phân 2 cụm: nền và vật thể
kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto').fit(X)
labels = kmeans.labels_.reshape(h, w)

# --- Xác định cụm nền ---
bg_label = np.bincount(labels.flatten()).argmax()
mask = (labels != bg_label).astype(np.uint8) * 255  # vùng vật thể = 255

# --- Làm mượt mask ---
mask = cv2.medianBlur(mask, 7)
mask = cv2.GaussianBlur(mask, (7, 7), 0)

# --- Chuẩn bị nền mới ---
bg = cv2.imread(background_path)
bg = cv2.resize(bg, (w, h))

# --- Hòa trộn ---
mask_3c = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR) / 255.0
result = (img * mask_3c + bg * (1 - mask_3c)).astype(np.uint8)

# --- Lưu kết quả ---
# cv2.imwrite(output_path, result)

# --- Hiển thị ---
cv2.imshow("Original", img)
cv2.imshow("Mask", mask)
cv2.imshow("Result", result)
cv2.waitKey(0)
cv2.destroyAllWindows()
